# サンプリング波形の異常検知 設計仕様書（第1版）

## 1. 文書の目的

本書は、1本センサのサンプリング波形に対して、**正常状態からの変化を異常として検知するための再現可能な設計仕様**を定義するものである。

本書で対象とする状況は、次の条件を満たす。

- データは各観測点ごとに「センサ値」「時間」「カム角度 $0\sim360$ 度」を持つ。
- 1周ごとに、ほぼ同じ機械動作が繰り返される。
- ただし各周回の観測は**不等間隔**であり、1周あたりの観測点数は **9〜12点程度** と少ない。
- 正常データは **300〜1000周程度** あり、正常状態は当面 **1種類** とみなす。
- 最終的な test データには **正常・異常の両方** を含み、異常ラベルは **1周ごと** に付いている。

本書では、ここまでの議論で出た疑問点と設計判断を反映し、次を明確にする。

1. どの前提条件を置くか。
2. なぜその特徴量を使うか。
3. 周波数成分の変化、振幅倍率だけの変化、位相ずれ、局所異常をどのように分けて扱うか。
4. どの数式で特徴量を計算し、どの数式で異常判定するか。
5. どのように可視化し、どのように評価するか。

---

## 2. 検知したい背景と設計要求

### 2.1 背景

対象は、周期動作をもつ機械の1本センサ波形である。

このとき現場で知りたいのは、単に「異常/正常」の2値だけではない。最低限、次が必要である。

- **いつ異常が出たか** が時系列で見えること。
- **何が原因で異常になったか** が見えること。
- 特に、**周波数成分が変化せず、振幅倍率だけが変化した場合も異常として検知できること**。
- 高次側や細かい変化については、厳密に「高周波成分の変化」と特定できなくてもよいが、**異常としては拾えること**。

### 2.2 設計要求

本仕様では、次の要求を満たすことを目標とする。

1. 1周ごとに1個の総合異常スコアを出せること。
2. 総合スコアの内訳として、少なくとも次の原因別スコアを持つこと。
   - 上下ずれ
   - 次数成分の振幅変化
   - 次数成分の位相変化
   - 低次の周波数成分変化
   - 局所・高次残差異常
3. 「どれか1本が危険なら異常」という判定ができること。
4. 1周ごとの時系列グラフと、正常分布との位置関係を見る散布図を持つこと。
5. 不等間隔・少数観測という条件でも、過度な補間に依存しないこと。

---

## 3. 採用した全体方針

### 3.1 1周を1サンプルとする

本仕様では、**カム角度が1周する区間を1サンプル**とする。

理由は次の通りである。

- 動作が1周ごとにほぼ繰り返される。
- 角度情報があるため、同じ機械状態を時間よりも角度で比べやすい。
- 異常ラベルも1周ごとに付いている。

### 3.2 時間軸ではなく角度軸を主に使う

時間軸そのままで比較すると、回転速度のゆらぎによって波形が横方向に伸び縮みして見える。そこで本仕様では、時間ではなく**角度基準**で1周の波形を表現する。

時間周波数 $f$ と回転基本周波数 $f_{\mathrm{rot}}$ の関係は、order $k$ を使うと

$$
f = k\,f_{\mathrm{rot}}
$$

と書ける。角度基準で表現する場合、時間周波数そのものではなく、**order（次数）成分**を見ることになる。

### 3.3 生波形の密な補間を主役にしない

1周あたりの観測点数が 9〜12 点しかないため、各周回を 256 点や 360 点へ細かく補間して、その補間波形を主役にする設計は採用しない。

理由は、増えた点の多くが観測ではなく推定になるためである。第1版では、**観測点を直接使って少数の周期係数へ要約する**方針を採る。

### 3.4 深い学習を第1候補にしない

自己符号化器などの深い学習は、将来の比較候補としてはあり得るが、第1版では採用しない。理由は次の通りである。

- 1周ごとの観測点が少ない。
- まずは何が変わったか説明しやすい設計が必要である。
- 正常データが比較的多いため、正常分布を明示的に学ぶ手法が使いやすい。

### 3.5 各段は個別 PCA ではなく、小次元のマハラノビス距離で判定する

各原因別スコアの入力次元は 2〜3 次元程度である。PCA は高次元の相関特徴を圧縮する時に有効だが、本件では各段の次元が低いため、第1版では各段をそのままロバストなマハラノビス距離で判定する。

---

## 4. 用語と記号

### 4.1 周回と観測

周回番号を $i=1,2,\dots,N$ とする。周回 $i$ の観測点数を $n_i$ とする。

周回 $i$ の $j$ 番目の観測を

- 角度: $\theta_{ij}$ [deg]
- センサ値: $y_{ij}$
- 時刻: $t_{ij}$

とする。ただし本設計の主変数は角度であり、時刻は可視化や区間対応のために保持する。

### 4.2 角度のラジアン化

三角関数に入れる角度はラジアンに直す。以降、

$$
\vartheta_{ij} = 2\pi\,\frac{\theta_{ij}}{360}
$$

を用いる。

### 4.3 周回境界

0 度の観測点が毎回存在する必要はない。周回境界は、連続観測において角度が大きく負方向に折り返す箇所、すなわち

$$
\theta_{m+1} - \theta_m < -\Delta_{\mathrm{wrap}}
$$

となる点を使って推定する。第1版では既定値として

$$
\Delta_{\mathrm{wrap}} = 180\;[\mathrm{deg}]
$$

を用いる。

---

## 5. データ分割仕様

### 5.1 分割の考え方

データは次の 3 群に分ける。

- **fit 用**: 正常のみ
- **threshold 用**: 正常のみ
- **test 用**: 正常 + 異常

### 5.2 各群の役割

fit 用は、正常分布の中心と広がりを学ぶために使う。

threshold 用は、各段のスコアのしきい値を決めるために使う。fit 用と分ける理由は、学習に使ったデータだけでしきい値を決めると、しきい値が甘くなる可能性があるためである。

test 用は、どちらにも使っていない最終確認用であり、正常に対する誤報と異常に対する検知の両方を評価する。

### 5.3 分割比率

第1版の既定値は

- fit: 60%
- threshold: 20%
- test: 20%

とする。

### 5.4 時系列順分割

データは原則として**時間順または周回順に前から後ろへ切る**。ランダム分割は第1版では採用しない。

---

## 6. 1周の周期モデル

### 6.1 モデル式

周回 $i$ のセンサ値は、角度 $\vartheta$ に対して次の 3 次までの周期モデルで表す。

$$
\hat y_i(\vartheta)
=
a_{0,i}
+ \sum_{k=1}^{3}\left(a_{k,i}\cos(k\vartheta)+b_{k,i}\sin(k\vartheta)\right)
$$

観測値との関係は

$$
y_{ij} = \hat y_i(\vartheta_{ij}) + r_{ij}
$$

であり、$r_{ij}$ は残差である。

### 6.2 行列表現

周回 $i$ の設計行列 $X_i \in \mathbb{R}^{n_i\times 7}$ を

$$
X_i =
\begin{bmatrix}
1 & \cos \vartheta_{i1} & \sin \vartheta_{i1} & \cos 2\vartheta_{i1} & \sin 2\vartheta_{i1} & \cos 3\vartheta_{i1} & \sin 3\vartheta_{i1}\\
1 & \cos \vartheta_{i2} & \sin \vartheta_{i2} & \cos 2\vartheta_{i2} & \sin 2\vartheta_{i2} & \cos 3\vartheta_{i2} & \sin 3\vartheta_{i2}\\
\vdots & \vdots & \vdots & \vdots & \vdots & \vdots & \vdots \\
1 & \cos \vartheta_{in_i} & \sin \vartheta_{in_i} & \cos 2\vartheta_{in_i} & \sin 2\vartheta_{in_i} & \cos 3\vartheta_{in_i} & \sin 3\vartheta_{in_i}
\end{bmatrix}
$$

とする。

係数ベクトルを

$$
\beta_i =
\begin{bmatrix}
a_{0,i} & a_{1,i} & b_{1,i} & a_{2,i} & b_{2,i} & a_{3,i} & b_{3,i}
\end{bmatrix}^{\top}
$$

とする。

### 6.3 係数推定

第1版では、周回 $i$ の係数は最小二乗で求める。

$$
\hat\beta_i = \arg\min_{\beta} \lVert y_i - X_i\beta \rVert_2^2
$$

実装では、行列が不安定な場合に備えて擬似逆行列を用い、

$$
\hat\beta_i = X_i^{+} y_i
$$

とする。

### 6.4 3 次までとした理由

1周あたりの観測点数が 9〜12 点と少ないため、次数を上げすぎると推定が不安定になる。第1版では、低次のなめらかな変化はモデル側、細かい変化は残差側で見る方針に従い、3 次までとする。

---

## 7. 周期モデルから導出する基本量

### 7.1 次数成分の振幅

各次数 $k=1,2,3$ について、振幅を

$$
A_{k,i} = \sqrt{a_{k,i}^2 + b_{k,i}^2}
$$

と定義する。

### 7.2 次数成分の位相

各次数 $k=1,2,3$ について、位相を

$$
\phi_{k,i} = \operatorname{atan2}(b_{k,i}, a_{k,i})
$$

と定義する。

### 7.3 残差

周回 $i$ の $j$ 番目観測に対する残差は

$$
r_{ij} = y_{ij} - \hat y_i(\vartheta_{ij})
$$

である。

### 7.4 全体 RMS

周回 $i$ の全体 RMS は

$$
\mathrm{RMS}_i = \sqrt{\frac{1}{n_i}\sum_{j=1}^{n_i} y_{ij}^2}
$$

とする。

### 7.5 残差 RMS

残差 RMS は

$$
\mathrm{RMS}^{(r)}_i = \sqrt{\frac{1}{n_i}\sum_{j=1}^{n_i} r_{ij}^2}
$$

とする。

### 7.6 残差の上位量

局所異常を見やすくするため、

$$
P95_i = \operatorname{quantile}_{0.95}\left(|r_{i1}|,\dots,|r_{in_i}|\right)
$$

$$
M_i = \max_j |r_{ij}|
$$

を定義する。

---

## 8. 監視段の設計と採用理由

第1版では、原因別に 5 本の監視段を持つ。

### 8.1 上下ずれ

#### 8.1.1 目的

- 波形全体が上にずれた / 下にずれた変化を拾う。
- 全体強度の変化も一部ここで補助的に見る。

#### 8.1.2 特徴量

$$
x^{(\mathrm{level})}_i =
\begin{bmatrix}
a_{0,i} \\
\mathrm{RMS}_i
\end{bmatrix}
$$

#### 8.1.3 採用理由

$a_0$ は波形の平均的な上下ずれを直接表す。$\mathrm{RMS}$ は全体の強さを反映する。どちらも観測点数が少なくても比較的安定して計算できる。

### 8.2 次数成分の振幅変化

#### 8.2.1 目的

- **周波数成分は変わらず、振幅倍率だけが変わる異常** を確実に拾う。
- 1〜3 次の絶対振幅の増減を拾う。

#### 8.2.2 特徴量

既定の特徴量は

$$
x^{(\mathrm{amp})}_i =
\begin{bmatrix}
A_{1,i} \\
A_{2,i} \\
A_{3,i}
\end{bmatrix}
$$

とする。拡張版では

$$
A_{\mathrm{sum},i}=A_{1,i}+A_{2,i}+A_{3,i}
$$

を追加してもよいが、第1版では用いない。

#### 8.2.3 採用理由

波形が単純に $c$ 倍されるだけなら

$$
a_{k,i}\to c a_{k,i}, \qquad b_{k,i}\to c b_{k,i}
$$

であるため、

$$
A_{k,i}\to c A_{k,i}
$$

となる。したがって、$A_1,A_2,A_3$ を監視していれば、**振幅倍率だけの変化**を直接検知できる。

### 8.3 次数成分の位相変化

#### 8.3.1 目的

- 横ずれ、山谷位置のずれを拾う。
- 機械的な位相ずれやタイミングずれに対応する。

#### 8.3.2 位相差の定義

位相は角度量なので、そのまま線形に引き算してはいけない。正常の代表位相を $\bar\phi_k$ とすると、位相差は

$$
\delta_{k,i} = \operatorname{wrap}(\phi_{k,i}-\bar\phi_k)
$$

とする。ここで $\operatorname{wrap}(\cdot)$ は角度差を $(-\pi,\pi]$ に巻き戻す演算であり、具体的には

$$
\operatorname{wrap}(x) = \left((x+\pi) \bmod 2\pi\right)-\pi
$$

と定義する。

#### 8.3.3 特徴量

$$
x^{(\mathrm{phase})}_i =
\begin{bmatrix}
\delta_{1,i} \\
\delta_{2,i} \\
\delta_{3,i}
\end{bmatrix}
$$

#### 8.3.4 採用理由

359 度と 1 度は近いので、位相は円環量として扱う必要がある。位相差を巻き戻して使うことで、横ずれを素直に表せる。

### 8.4 低次の周波数成分変化

#### 8.4.1 目的

- 1〜3 次のどこに成分が多いか、その**配分の変化**を拾う。
- 振幅倍率だけの変化とは区別して、低次の周波数成分変化を見たい。

#### 8.4.2 特徴量の定義

第1版では、各次数のエネルギー配分を

$$
p_{k,i} = \frac{A_{k,i}^2}{A_{1,i}^2 + A_{2,i}^2 + A_{3,i}^2 + \varepsilon}
$$

で定義する。ここで数値安定化のため、既定値として

$$
\varepsilon = 10^{-12}
$$

を用いる。

特徴量は

$$
x^{(\mathrm{freq})}_i =
\begin{bmatrix}
p_{1,i} \\
p_{2,i} \\
p_{3,i}
\end{bmatrix}
$$

とする。

#### 8.4.3 採用理由

$A_1,A_2,A_3$ の絶対値だけを見ると、振幅倍率だけの変化と、次数間の配分変化が混ざってしまう。配分 $p_k$ を使うと、全体が同じ倍率で増減しても配分はほぼ不変であり、**低次の周波数成分変化**を見やすい。

### 8.5 局所・高次残差異常

#### 8.5.1 目的

- 3 次までで説明できない細かい変化を拾う。
- スパイクを拾う。
- 高次側の変化を、厳密に分類できなくても異常として拾う。

#### 8.5.2 特徴量

$$
x^{(\mathrm{resid})}_i =
\begin{bmatrix}
\mathrm{RMS}^{(r)}_i \\
P95_i \\
M_i
\end{bmatrix}
$$

#### 8.5.3 採用理由

高次側の成分や局所異常は、3 次までの低次モデルには入りにくい。そのため、それらは残差に落ちる。よって、残差系特徴量を監視すれば、**高周波成分変化だと特定はできなくても、異常としては拾える**。

---

## 9. 品質指標

### 9.1 目的

0 度の点が毎回あるとは限らず、また観測角度が一部に偏ると係数推定が不安定になる。そのため、異常スコアとは別に**判定の信頼性**を示す品質指標を持つ。

### 9.2 定義

#### 9.2.1 観測点数

$$
q^{(\mathrm{nobs})}_i = n_i
$$

#### 9.2.2 角度カバー率

周回 $i$ の観測角度を昇順に並べたものを

$$
\theta^{\uparrow}_{i,(1)} \le \theta^{\uparrow}_{i,(2)} \le \cdots \le \theta^{\uparrow}_{i,(n_i)}
$$

とする。角度差列を

$$
g_{i,m} = \theta^{\uparrow}_{i,(m+1)} - \theta^{\uparrow}_{i,(m)} \quad (m=1,\dots,n_i-1)
$$

$$
g_{i,n_i} = 360 - \theta^{\uparrow}_{i,(n_i)} + \theta^{\uparrow}_{i,(1)}
$$

と定義する。

最大角度ギャップは

$$
G_i = \max_m g_{i,m}
$$

とする。

角度カバー率は

$$
C_i = 1 - \frac{G_i}{360}
$$

とする。

### 9.3 使い方

品質指標は異常判定そのものには入れない。代わりに、

- $n_i$ が極端に少ない
- $G_i$ が極端に大きい
- $C_i$ が極端に低い

ときは「判定保留」や「低信頼」として別表示する。

---

## 10. 正常分布の学習

### 10.1 正常代表量

各監視段 $m \in \{\mathrm{level}, \mathrm{amp}, \mathrm{phase}, \mathrm{freq}, \mathrm{resid}\}$ に対し、fit 用正常データから平均ベクトル $\mu^{(m)}$ とロバスト共分散行列 $\Sigma^{(m)}$ を求める。

第1版では、ロバスト共分散推定器として Minimum Covariance Determinant 系を想定する。

### 10.2 位相段の中心

位相差段では、まず fit 用正常データの各次数位相の代表値 $\bar\phi_k$ を求める。その上で各周回の $\delta_{k,i}$ を計算する。

実装上は、正常位相群 $\{\phi_{k,i}\}$ に対して円環平均を用いる。具体的には

$$
\bar\phi_k = \operatorname{atan2}\left(\frac{1}{N_{\mathrm{fit}}}\sum_i \sin \phi_{k,i},\; \frac{1}{N_{\mathrm{fit}}}\sum_i \cos \phi_{k,i}\right)
$$

とする。

---

## 11. 各段の異常スコア

### 11.1 上下ずれスコア

$$
s^{(\mathrm{level})}_i =
\left(x^{(\mathrm{level})}_i - \mu^{(\mathrm{level})}\right)^{\top}
\left(\Sigma^{(\mathrm{level})}\right)^{-1}
\left(x^{(\mathrm{level})}_i - \mu^{(\mathrm{level})}\right)
$$

### 11.2 次数成分の振幅変化スコア

$$
s^{(\mathrm{amp})}_i =
\left(x^{(\mathrm{amp})}_i - \mu^{(\mathrm{amp})}\right)^{\top}
\left(\Sigma^{(\mathrm{amp})}\right)^{-1}
\left(x^{(\mathrm{amp})}_i - \mu^{(\mathrm{amp})}\right)
$$

### 11.3 次数成分の位相変化スコア

第1版では、位相差ベクトルにもロバスト共分散を適用して

$$
s^{(\mathrm{phase})}_i =
\left(x^{(\mathrm{phase})}_i - \mu^{(\mathrm{phase})}\right)^{\top}
\left(\Sigma^{(\mathrm{phase})}\right)^{-1}
\left(x^{(\mathrm{phase})}_i - \mu^{(\mathrm{phase})}\right)
$$

とする。なお、$\mu^{(\mathrm{phase})}$ は位相差空間での平均ベクトルであり、fit 用正常データから求める。

### 11.4 低次の周波数成分変化スコア

$$
s^{(\mathrm{freq})}_i =
\left(x^{(\mathrm{freq})}_i - \mu^{(\mathrm{freq})}\right)^{\top}
\left(\Sigma^{(\mathrm{freq})}\right)^{-1}
\left(x^{(\mathrm{freq})}_i - \mu^{(\mathrm{freq})}\right)
$$

### 11.5 局所・高次残差異常スコア

$$
s^{(\mathrm{resid})}_i =
\left(x^{(\mathrm{resid})}_i - \mu^{(\mathrm{resid})}\right)^{\top}
\left(\Sigma^{(\mathrm{resid})}\right)^{-1}
\left(x^{(\mathrm{resid})}_i - \mu^{(\mathrm{resid})}\right)
$$

---

## 12. しきい値と総合判定

### 12.1 各段しきい値

threshold 用正常データに対して各段スコアを計算し、その上側分位点をしきい値とする。

第1版の既定値は

$$
q_{\mathrm{thr}} = 0.995
$$

すなわち 99.5 パーセンタイルとする。

各段のしきい値を

$$
\tau_{\mathrm{level}},\;\tau_{\mathrm{amp}},\;\tau_{\mathrm{phase}},\;\tau_{\mathrm{freq}},\;\tau_{\mathrm{resid}}
$$

とする。

### 12.2 正規化スコア

$$
u_{\mathrm{level},i} = \frac{s^{(\mathrm{level})}_i}{\tau_{\mathrm{level}}}
$$

$$
u_{\mathrm{amp},i} = \frac{s^{(\mathrm{amp})}_i}{\tau_{\mathrm{amp}}}
$$

$$
u_{\mathrm{phase},i} = \frac{s^{(\mathrm{phase})}_i}{\tau_{\mathrm{phase}}}
$$

$$
u_{\mathrm{freq},i} = \frac{s^{(\mathrm{freq})}_i}{\tau_{\mathrm{freq}}}
$$

$$
u_{\mathrm{resid},i} = \frac{s^{(\mathrm{resid})}_i}{\tau_{\mathrm{resid}}}
$$

### 12.3 総合スコア

ユーザー要求は「どれか1本で異常」であるため、総合は和ではなく最大値とする。

$$
u_{\mathrm{total},i} =
\max\left(
\nu_{\mathrm{level},i},
\nu_{\mathrm{amp},i},
\nu_{\mathrm{phase},i},
\nu_{\mathrm{freq},i},
\nu_{\mathrm{resid},i}
\right)
$$

### 12.4 1周単位の異常判定

周回 $i$ の異常判定は

$$
\nu_{\mathrm{total},i} > 1
$$

なら異常、そうでなければ正常とする。

### 12.5 主因ラベル

異常時には、最大だった内訳スコアの段名を主因とする。すなわち

$$
\operatorname{cause}_i = \arg\max_{m \in \{\mathrm{level},\mathrm{amp},\mathrm{phase},\mathrm{freq},\mathrm{resid}\}} \nu_{m,i}
$$

とする。

この設計の利点は、「どれか1本が危険なら異常」という要求をそのまま満たし、かつ**何が主因だったかがそのまま読める**ことである。

---

## 13. 可視化仕様

### 13.1 主画面: 時系列グラフ

本番監視で主役となる画面は、周回番号または時刻を横軸にした時系列グラフである。表示する系列は次の 6 本とする。

1. 総合スコア $\nu_{\mathrm{total}}$
2. 上下ずれ $\nu_{\mathrm{level}}$
3. 次数成分の振幅変化 $\nu_{\mathrm{amp}}$
4. 次数成分の位相変化 $\nu_{\mathrm{phase}}$
5. 低次の周波数成分変化 $\nu_{\mathrm{freq}}$
6. 局所・高次残差異常 $\nu_{\mathrm{resid}}$

各段には $1.0$ の水平しきい値線を引く。

### 13.2 補助画面: 散布図

散布図は開発・説明用であり、正常分布と test 点の位置関係を見るために使う。各散布図では、fit 用正常データのロバスト共分散から得た楕円境界を重ねる。

推奨する表示は次の通りである。

- 上下ずれ: $(a_0, \mathrm{RMS})$
- 次数成分の振幅変化: $(A_1, A_2)$, $(A_1, A_3)$, $(A_2, A_3)$
- 次数成分の位相変化: $(\delta_1, \delta_2)$, $(\delta_1, \delta_3)$, $(\delta_2, \delta_3)$
- 低次の周波数成分変化: $(p_1, p_2)$, $(p_1, p_3)$, $(p_2, p_3)$
- 局所・高次残差異常: $(\mathrm{RMS}^{(r)}, M)$

### 13.3 品質指標の表示

品質指標は異常スコアと別段で表示する。特に、最大角度ギャップ $G_i$ と角度カバー率 $C_i$ は、スコア異常との区別に重要である。

---

## 14. 評価仕様

### 14.1 test の前提

test には正常と異常の両方が含まれ、ラベルは 1 周ごとに付与されている。

### 14.2 最低限の評価指標

第1版で必須とする評価は次の通りである。

#### 14.2.1 2値評価

- 混同行列
- 異常クラスの再現率
- 異常クラスの適合率
- F1

#### 14.2.2 種類別評価

異常ラベルが種類を持つ場合、少なくとも次の種類別再現率を出す。

- 上下ずれ
- 位相ずれ
- 振幅倍率変化
- 低次周波数成分変化
- 局所異常 / 高次側異常

#### 14.2.3 スコア曲線評価

総合スコア $\nu_{\mathrm{total}}$ に対して、PR 曲線と PR-AUC を計算する。

### 14.3 解析ログ

異常と判定された周回については、次をログとして残す。

- 周回番号
- 総合スコア
- 内訳 5 本
- 主因ラベル
- 品質指標

---

## 15. 実装手順

### 15.1 入力整形

1. 元データを時間順に並べる。
2. 角度折り返しを使って周回境界を切る。
3. 各周回ごとに $(\theta_{ij}, y_{ij}, t_{ij})$ を保持する。

### 15.2 データ分割

1. 正常周回を時系列順に fit / threshold / test-normal に分ける。
2. 異常周回を test-anomaly に入れる。
3. test は test-normal と test-anomaly を結合する。

### 15.3 周回ごとの特徴量計算

1. 各周回で設計行列 $X_i$ を作る。
2. 擬似逆で $\hat\beta_i$ を求める。
3. $A_k$, $\phi_k$, $r_{ij}$ を計算する。
4. 5 段の特徴量ベクトルを作る。
5. 品質指標を計算する。

### 15.4 fit

1. fit 用正常データで、各段の平均ベクトルとロバスト共分散を求める。
2. 位相代表値 $\bar\phi_k$ を求める。

### 15.5 threshold

1. threshold 用正常データに対して各段スコアを計算する。
2. 各段の 99.5 パーセンタイルをしきい値とする。

### 15.6 test

1. test データに対して各段スコアを計算する。
2. 正規化スコアと総合スコアを求める。
3. 1 周単位の異常判定を行う。
4. 主因ラベルを付ける。
5. 評価指標と可視化を出力する。

---

## 16. 本仕様で採用しなかった案と理由

### 16.1 密な補間波形を主役にする案

不採用理由は、1周あたりの観測点数が少なく、補間点の多くが推定値になるためである。

### 16.2 各段で個別 PCA をかける案

不採用理由は、各段の特徴量次元が低く、そのまま距離判定した方が自然で解釈しやすいためである。

### 16.3 Isolation Forest を第1候補にする案

比較候補としては有用だが、第1版では正常 1 モード・低次元・楕円可視化との相性から、マハラノビス距離を本命とした。

### 16.4 高次 order まで直接監視する案

観測点数 9〜12 点に対して高次まで直接推定すると不安定になりやすい。第1版では、低次はモデル側、高次や細かい変化は残差側で拾う設計にとどめる。

---

## 17. 本仕様で明確に答えた疑問点

### 17.1 0 度の点が毎回ないが問題ないか

問題ない。必要なのは各観測点の角度情報であり、0 度そのものの観測は必須ではない。ただし、角度カバーが悪い周回は品質低下として別管理する。

### 17.2 距離法だと時系列で見えないのではないか

距離も 1 周ごとに 1 個の数になるので、時系列グラフ化できる。散布図は補助表示であり、本番の主画面は管理図型の時系列でよい。

### 17.3 周波数成分の変化はどこで見るのか

低次の周波数成分変化は $p_1,p_2,p_3$ による配分変化スコアで見る。高次側の変化は残差側に落ちるので、局所・高次残差異常で拾う。

### 17.4 振幅倍率だけが変わる異常は検知できるか

できる。$A_1,A_2,A_3$ は振幅倍率変化に比例して変化するため、次数成分の振幅変化段で拾える。

### 17.5 高周波側も今の設計でわかるのではないか

異常としては気づける可能性があるが、高周波成分変化と特定できるとは限らない。第1版では「異常として拾えればよい」という整理を採る。

---

## 18. 第1版の固定パラメータ一覧

| 項目 | 記号 | 既定値 |
|---|---:|---:|
| モデル次数上限 | $K$ | 3 |
| 周回折り返し判定角度差 | $\Delta_{\mathrm{wrap}}$ | $180\,\mathrm{deg}$ |
| 配分計算の安定化項 | $\varepsilon$ | $10^{-12}$ |
| しきい値分位点 | $q_{\mathrm{thr}}$ | 0.995 |
| データ分割 | fit / threshold / test | 60 / 20 / 20 |

---

## 19. 参考文献

1. Lomb, N. R. (1976). Least-squares frequency analysis of unequally spaced data. *Astrophysics and Space Science*.
2. Zechmeister, M., & Kürster, M. (2009). The generalised Lomb-Scargle periodogram. *A&A*.
3. Astropy Documentation: Lomb-Scargle Periodograms.
4. Brüel & Kjær. *Order Tracking Analysis*.
5. scikit-learn Documentation: EllipticEnvelope / robust covariance / outlier detection.
6. NIST/SEMATECH e-Handbook of Statistical Methods: Multivariate Control Charts.

本仕様における不等間隔データの周期モデル、order 成分の振幅・位相、ロバスト距離による異常判定の考え方は、上記の文献・資料に沿って整理している。
